**1. ライブラリのインポートとデータ確認**

まずは分析に必要なライブラリを読み込み、Kaggle環境内の入力ファイルを確認する。

In [17]:
# ライブラリインポート
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import os

# 入力データのファイルパスを確認
# データの階層構造を把握するために実行
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/tounyoubyou/playground-series-s5e12/sample_submission.csv
/kaggle/input/tounyoubyou/playground-series-s5e12/train.csv
/kaggle/input/tounyoubyou/playground-series-s5e12/test.csv
/kaggle/input/3tounyoubyou/sample_submission.csv
/kaggle/input/3tounyoubyou/train.csv
/kaggle/input/3tounyoubyou/test.csv


**2. データの読み込み**

学習データ、テストデータ、および提出用のサンプルファイルを読み込む。

In [18]:
# データの読み込み
train = pd.read_csv("/kaggle/input/3tounyoubyou/train.csv")
test  = pd.read_csv("/kaggle/input/3tounyoubyou/test.csv")
sample= pd.read_csv("/kaggle/input/3tounyoubyou/sample_submission.csv")

# 提出形式（sample_submission）の確認
display(sample.head())

,id,diagnosed_diabetes
0,700000,0
1,700001,0
2,700002,0
3,700003,0
4,700004,0


**3. データの全体像の把握**

データがどんな形状をしているのか確認し、欠損値などデータの補完や前処理が必要かどうかも確認する。

In [19]:
print("===== train.csv shape =====")
print(train.shape)

print("\n===== test.csv shape =====")
print(test.shape)


print("\n===== train.csv head =====")
display(train.head())

print("\n===== test.csv head =====")
display(test.head())


print("\n===== train columns =====")
print(train.columns.tolist())

print("\n===== test columns =====")
print(test.columns.tolist())


print("\n===== train info =====")
train.info()

print("\n===== test info =====")
test.info()

===== train.csv shape =====
(700000, 26)

===== test.csv shape =====
(300000, 25)

===== train.csv head =====


,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0



===== test.csv head =====


,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,triglycerides,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history
0,700000,45,4,100,4.3,6.8,6.2,25.5,0.84,123,...,111,Female,White,Highschool,Middle,Former,Employed,0,0,0
1,700001,35,1,87,3.5,4.6,9.0,28.6,0.88,120,...,145,Female,White,Highschool,Middle,Never,Unemployed,0,0,0
2,700002,45,1,61,7.6,6.8,7.0,28.5,0.94,112,...,184,Male,White,Highschool,Low,Never,Employed,0,0,0
3,700003,55,2,81,7.3,7.3,5.0,26.9,0.91,114,...,128,Male,White,Graduate,Middle,Former,Employed,0,0,0
4,700004,77,2,29,7.3,7.6,8.5,22.0,0.83,131,...,133,Male,White,Graduate,Low,Current,Unemployed,0,0,0



===== train columns =====
['id', 'age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'diagnosed_diabetes']

===== test columns =====
['id', 'age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status', 'family_history_diabetes', 'hypertension_history', 'ca

In [20]:
# カテゴリ変数を数値化(ワンホットエンコーディング)
train1 = pd.get_dummies(train, dtype=np.int8)

train1.head()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,income_level_Lower-Middle,income_level_Middle,income_level_Upper-Middle,smoking_status_Current,smoking_status_Former,smoking_status_Never,employment_status_Employed,employment_status_Retired,employment_status_Student,employment_status_Unemployed
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,1,0,0,1,0,0,1,0,0,0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,0,0,1,0,0,1,1,0,0,0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,1,0,0,0,0,1,0,1,0,0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,1,0,0,1,0,0,1,0,0,0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,0,0,1,0,0,1,0,1,0,0


In [21]:
print(train1.columns.tolist())

['id', 'age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'diagnosed_diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'ethnicity_Asian', 'ethnicity_Black', 'ethnicity_Hispanic', 'ethnicity_Other', 'ethnicity_White', 'education_level_Graduate', 'education_level_Highschool', 'education_level_No formal', 'education_level_Postgraduate', 'income_level_High', 'income_level_Low', 'income_level_Lower-Middle', 'income_level_Middle', 'income_level_Upper-Middle', 'smoking_status_Current', 'smoking_status_Former', 'smoking_status_Never', 'employment_status_Employed', 'employment_status_Retired', 'employment_status_Student', 'employment_status_Unemployed']


In [22]:
# train1を説明変数と目的変数に分ける
X = train1.drop("diagnosed_diabetes", axis=1)
y = train1["diagnosed_diabetes"]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**4. モデルの構築と学習**

まずは最もシンプルなアルゴリズムの一つである「ロジスティック回帰」を使用する。
今回はmax_iterを1000に設定する。

In [23]:
# ロジスティック回帰
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

**5. モデルの評価**

検証データ（X_valid, y_valid）を用いて、モデルの性能を確認する。

In [24]:
# 作成したモデルで予測
pred = model.predict(X_valid)

# 予測精度の確認
print(classification_report(y_valid, pred))

accuracy = accuracy_score(y_valid, pred)
print("validation accuracy:", accuracy)

pred_proba = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, pred_proba)
print("Validation ROC-AUC:", auc)

              precision    recall  f1-score   support

         0.0       0.59      0.23      0.33     52629
         1.0       0.66      0.90      0.76     87371

    accuracy                           0.65    140000
   macro avg       0.62      0.57      0.55    140000
weighted avg       0.63      0.65      0.60    140000

validation accuracy: 0.6501
Validation ROC-AUC: 0.6524551552610036


In [25]:
# カテゴリ変数を数値化(ワンホットエンコーディング)
X_test = pd.get_dummies(test, dtype=np.int8)
X_test.head()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,income_level_Lower-Middle,income_level_Middle,income_level_Upper-Middle,smoking_status_Current,smoking_status_Former,smoking_status_Never,employment_status_Employed,employment_status_Retired,employment_status_Student,employment_status_Unemployed
0,700000,45,4,100,4.3,6.8,6.2,25.5,0.84,123,...,0,1,0,0,1,0,1,0,0,0
1,700001,35,1,87,3.5,4.6,9.0,28.6,0.88,120,...,0,1,0,0,0,1,0,0,0,1
2,700002,45,1,61,7.6,6.8,7.0,28.5,0.94,112,...,0,0,0,0,0,1,1,0,0,0
3,700003,55,2,81,7.3,7.3,5.0,26.9,0.91,114,...,0,1,0,0,1,0,1,0,0,0
4,700004,77,2,29,7.3,7.6,8.5,22.0,0.83,131,...,0,0,0,1,0,0,0,0,0,1


**6. 提出用ファイルの作成**

テストデータに対して予測を行い、提出用のCSVファイルを出力する。

In [26]:
# submission作成
y_test_proba = model.predict_proba(X_test)[:, 1]

submission = sample.copy()
submission["diagnosed_diabetes"] = y_test_proba

submission.to_csv("01_submission.csv", index=False)
display(submission.head())

,id,diagnosed_diabetes
0,700000,0.591911
1,700001,0.635675
2,700002,0.703804
3,700003,0.662838
4,700004,0.765017
